In [0]:
df=spark.read.option("header","true").option("inferSchema","true").csv("/Volumes/for_external/default/test_external/Employee_Attrition.csv")


In [0]:
display(df)

In [0]:
filtered_df = df.filter((df["Department"] == "Research & Development") & (df["JobRole"] == "Manager"))
display(filtered_df)

In [0]:
from pyspark.sql.functions import col, when, upper, concat_ws, year, month, lit, round

# Example heavy transformations:
transformed_df = (
    df
    # 1. Create a new column 'IsSenior' based on 'Age'
    .withColumn("IsSenior", when(col("Age") >= 50, lit(1)).otherwise(lit(0)))
    # 2. Uppercase the 'Department' column
    .withColumn("Department_Upper", upper(col("Department")))
    # 3. Concatenate 'EmployeeNumber' and 'JobRole' into a new column
    .withColumn("Emp_Job", concat_ws("_", col("EmployeeNumber"), col("JobRole")))
    # 4. Calculate 'YearsAtCompany' bucket
    .withColumn("TenureBucket", 
                when(col("YearsAtCompany") < 3, "Junior")
                .when((col("YearsAtCompany") >= 3) & (col("YearsAtCompany") < 7), "Mid")
                .otherwise("Senior"))
    # 5. Round 'MonthlyIncome' to nearest thousand
    .withColumn("MonthlyIncomeRounded", round(col("MonthlyIncome")/1000)*1000)
    # 6. Create a flag for high attrition risk
    .withColumn("HighAttritionRisk", 
                when((col("Attrition") == "Yes") & (col("OverTime") == "Yes") & (col("JobSatisfaction") < 2), lit(1)).otherwise(lit(0)))
    # 7. Select and reorder columns
    .select(
        "EmployeeNumber", "Emp_Job", "Department_Upper", "IsSenior", "TenureBucket",
        "MonthlyIncomeRounded", "HighAttritionRisk", "Attrition", "OverTime", "JobSatisfaction"
    )
)

display(transformed_df)

In [0]:
high_risk_attrition_df = transformed_df.filter(col("HighAttritionRisk") == 1)
high_risk_attrition_df.write.format("delta").mode("overwrite").save("/Volumes/for_external/default/test_external/high_risk_attrition_employee")

In [0]:
display(spark.sql("DESCRIBE HISTORY high_risk_attrition_employee"))


In [0]:
delta_df = spark.read.format("delta").load("/Volumes/for_external/default/test_external/high_risk_attrition_employee")
display(delta_df)